# Compare Detection frames directly

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/visual/Document_Visual_Compare_Images.ipynb)

In [ ]:
!pip install --upgrade --no-cache-dir git+https://github.com/IchigoMazone/klygo.git@v2-release

In [ ]:
import tempfile
from PIL import Image
from klygo import files, media, outputs, visual
tutorial_workspace = tempfile.TemporaryDirectory()
root = files.path(tutorial_workspace.name)
first = outputs.Detection(
    [outputs.Box((40, 50, 140, 170), label="cat", score=0.9)],
    source=Image.new("RGB", (320, 240), "lightgray"),
    source_id="image-a", image_size=(320, 240),
)
second = outputs.Detection(
    [outputs.Box((70, 40, 190, 150), label="dog", score=0.8)],
    source=Image.new("RGB", (280, 200), "lightblue"),
    source_id="image-b", image_size=(280, 200),
)


## Jump to an API

[visual.compare_images](#visualcompare_images)

### visual.compare_images

```python
compare_images(first, second, *, mode="show", output_path=None,
               titles=("A", "B"), show_scores=True, backend="auto", overwrite=False)
```

- `first`, `second`: Detection frames, each providing image source and annotations. Select a frame first when starting with Detections.
- `mode`: show (default) or save.
- `output_path`: required for save; not accepted for other modes. String and os.PathLike paths are supported.
- `titles`: two strings, one for each panel.
- `show_scores`: include confidence in labels.
- `backend`: display backend forwarded to show_image for show mode.
- `overwrite`: allow replacing an output file; defaults to False.
- Returns: a new RGB PIL preview in every mode, preserving source scales and aspect ratios.
- Errors: TypeError for invalid inputs/options; ValueError for invalid mode/path usage, title count, or source size mismatch; RuntimeError for unavailable source pixels; FileNotFoundError for missing source files; FileExistsError for an existing destination without overwrite.

**Basic use**: build a comparison from two frames, with no separate image arguments.


In [ ]:
preview = visual.compare_images(first, second, mode="show", titles=("Image A", "Image B"))
print(preview.size)


**Practical variation**: save directly without opening a viewer.

In [ ]:
preview = visual.compare_images(
    first, second, mode="save", output_path=root / "comparison.png",
    titles=("Image A", "Image B"), overwrite=True,
)
print(media.info(root / "comparison.png")["size"])


**Expected error and recovery**: save mode requires a destination.

In [ ]:
try:
    visual.compare_images(first, second, mode="save")
except ValueError as error:
    print(error)
visual.compare_images(first, second, mode="save", output_path=root / "recovered.png", overwrite=True)


## Put it together

In [ ]:
alternative = outputs.Detection(
    [outputs.Box((50, 55, 150, 175), label="dog", score=0.7)],
    source=first.image, source_id=first.source_id, image_size=first.image_size,
)
preview = visual.compare_images(
    first, alternative, mode="save", output_path=root / "same-image.png",
    titles=("Model A", "Model B"), show_scores=False, overwrite=True,
)
print(preview.size)


## Remember

Each Detection owns the connection to its source pixels and annotations. It must
have an in-memory source or a readable source_path. Video sources use frame_index.
Detection.image provides an independent source image; file readers are closed
after pixels have been copied. No matching, merging, or metrics are performed.
Save mode never opens a viewer. Source images and annotations are
unchanged. Different image sources and dimensions are allowed. Colors follow
draw_bboxes independently per panel. Desktop auto display can open a viewer;
OpenCV waits for a key.

[Overview](Document_Visual_Overview.ipynb) · [Draw boxes](Document_Visual_Draw_Bboxes.ipynb) · [Display](Document_Visual_Show_Image.ipynb) · [Outputs](../outputs/Document_Outputs_Overview.ipynb) · [Media](../media/Document_Media_Overview.ipynb)


In [ ]:
tutorial_workspace.cleanup()